In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score


In [ ]:
!pip install catboost
from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [ ]:
print(train.shape)
print(test.shape)

(439140, 16)
(188165, 15)


In [ ]:
train = train.dropna(subset=["PitNextLap"])

In [ ]:
target = "PitNextLap"

features = [col for col in train.columns if col not in ["id", target]]

In [ ]:
def add_features(df):
    df = df.copy()

    df["DegradationRate"] = df["Cumulative_Degradation"] / (df["TyreLife"] + 1)

    df["TyrePressure"] = df["TyreLife"] / (df["LapNumber"] + 1)

    df["IsLateRace"] = (df["RaceProgress"] > 0.7).astype(int)

    df["PositionPressure"] = df["Position"] * df["Position_Change"]

    df["LapEfficiency"] = df["LapTime (s)"] / (df["LapNumber"] + 1)

    return df

In [ ]:
train = add_features(train)
test = add_features(test)

In [ ]:
features = [col for col in train.columns if col not in ["id", target]]

In [ ]:
X = train[features]
y = train[target]

X_test = test[features]

In [ ]:
groups = train["Race"] + "_" + train["Driver"]

In [ ]:
gkf = GroupKFold(n_splits=5)

scores = []

for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"\n========== FOLD {fold+1} ==========")

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3,
        eval_metric="AUC",
        verbose=200
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        cat_features=["Driver", "Compound", "Race"],
        use_best_model=True
    )

    preds = model.predict_proba(X_valid)[:, 1]

    score = roc_auc_score(y_valid, preds)
    print("Fold AUC:", score)

    scores.append(score)


========== FOLD 1 ==========
0:	test: 0.9142555	best: 0.9142555 (0)	total: 1.2s	remaining: 19m 55s
200:	test: 0.9395178	best: 0.9395178 (200)	total: 2m 25s	remaining: 9m 37s
400:	test: 0.9427968	best: 0.9427968 (400)	total: 4m 30s	remaining: 6m 43s
600:	test: 0.9446745	best: 0.9446745 (600)	total: 6m 27s	remaining: 4m 17s
800:	test: 0.9456460	best: 0.9456460 (800)	total: 8m 23s	remaining: 2m 5s
999:	test: 0.9462821	best: 0.9462821 (999)	total: 10m 18s	remaining: 0us

bestTest = 0.9462820748
bestIteration = 999

Fold AUC: 0.9462820748051248

========== FOLD 2 ==========
0:	test: 0.9142256	best: 0.9142256 (0)	total: 514ms	remaining: 8m 33s
200:	test: 0.9407192	best: 0.9407192 (200)	total: 2m 8s	remaining: 8m 28s
400:	test: 0.9439960	best: 0.9439960 (400)	total: 4m 3s	remaining: 6m 3s
600:	test: 0.9457628	best: 0.9457628 (600)	total: 5m 56s	remaining: 3m 56s
800:	test: 0.9467492	best: 0.9467496 (799)	total: 7m 53s	remaining: 1m 57s
999:	test: 0.9474468	best: 0.9474477 (996)	total: 9m 48s

In [ ]:
print("\n========== RESULTS ==========")
print("Mean AUC:", np.mean(scores))
print("Std AUC:", np.std(scores))


========== RESULTS ==========
Mean AUC: 0.9471165014567733
Std AUC: 0.0007913148035265229


In [ ]:
final_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3,
    eval_metric="AUC",
    verbose=200
)

final_model.fit(
    X, y,
    cat_features=["Driver", "Compound", "Race"]
)

0:	total: 1.7s	remaining: 28m 16s
200:	total: 2m 53s	remaining: 11m 28s
400:	total: 5m 29s	remaining: 8m 12s
600:	total: 8m 12s	remaining: 5m 26s
800:	total: 10m 52s	remaining: 2m 42s
999:	total: 13m 22s	remaining: 0us


CatBoostClassifier(depth=8, eval_metric='AUC', iterations=1000, l2_leaf_reg=3, learning_rate=0.03, verbose=200)

In [ ]:
test_preds = final_model.predict_proba(X_test)[:, 1]

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "PitNextLap": test_preds
})

submission.to_csv("submission_v2.csv", index=False)